# Customer Churn Prediction with Tree-Based Models

## Project Overview
This project explores customer churn in a telecom dataset and builds tree-based machine learning models to predict whether a customer is likely to leave the service.

The notebook is structured as a portfolio-style case study rather than a course exercise. It focuses on:
- data cleaning and exploratory analysis
- churn drivers and customer segmentation
- tree-based modeling with hyperparameter tuning
- model comparison and interpretation

## Business Goal
Customer churn directly affects recurring revenue. The main goal of this analysis is to identify patterns associated with churn and develop an interpretable classification workflow that can support retention-focused decisions.

## Models Used
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier

## Workflow
1. Load and clean the data  
2. Explore the target and key features  
3. Create simple tenure-based customer cohorts  
4. Prepare encoded features for modeling  
5. Train and tune multiple tree-based models  
6. Compare performance using classification metrics  
7. Interpret the most useful features

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## Data Loading
The notebook expects the dataset to be placed in the `data/` folder:

`data/Telco-Customer-Churn.csv`

If your dataset is stored elsewhere, update the path in the next cell.

In [ ]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df.head()

## Initial Inspection
A quick inspection helps verify the schema, data types, and possible cleaning issues before any modeling begins.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.info()

In [ ]:
df.describe(include="all").T.head(10)

## Data Cleaning
In the original telecom churn dataset, `TotalCharges` is often stored as text because of blank string values.  
We convert it to numeric and remove rows where the value cannot be recovered.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"]).copy()

print(df.isna().sum().sort_values(ascending=False).head())
df.head()

## Target Variable
The target variable is `Churn`. Before building models, it is useful to understand whether the classes are balanced.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Class Distribution of Churn")
plt.show()

churn_share = df["Churn"].value_counts(normalize=True).mul(100).round(2)
churn_share

**Interpretation:**  
This plot shows the proportion of customers who stayed versus those who churned.  
If the classes are moderately imbalanced, accuracy alone may not be sufficient, so recall and F1-score become especially important.

## Exploratory Data Analysis
The next charts focus on churn behavior across contract type, tenure, and billing patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[0])
axes[0].set_title("Monthly Charges by Churn")

sns.boxplot(data=df, x="Churn", y="TotalCharges", ax=axes[1])
axes[1].set_title("Total Charges by Churn")

plt.tight_layout()
plt.show()

**Interpretation:**  
Charge distributions often reveal whether churn is more common among customers with higher monthly costs or among customers with shorter total spending histories.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.show()

**Interpretation:**  
Contract structure is usually one of the strongest churn signals. Month-to-month customers often show much higher churn than customers on longer commitments.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, multiple="stack")
plt.title("Tenure Distribution by Churn")
plt.show()

**Interpretation:**  
Tenure helps capture customer lifecycle behavior. Early churn is especially important from a business perspective because it may indicate weak onboarding or low early engagement.

## Feature Engineering: Tenure Cohorts
To make churn behavior easier to interpret, we group customers into broader tenure segments.

In [ ]:
def tenure_to_cohort(tenure):
    if tenure <= 12:
        return "0-12 Months"
    elif tenure <= 24:
        return "12-24 Months"
    elif tenure <= 48:
        return "24-48 Months"
    return "Over 48 Months"

df["TenureCohort"] = df["tenure"].apply(tenure_to_cohort)
df[["tenure", "TenureCohort"]].head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="TenureCohort", hue="Churn")
plt.title("Churn by Tenure Cohort")
plt.xticks(rotation=15)
plt.show()

**Interpretation:**  
This segmentation makes it easier to see whether churn is concentrated among newer customers or remains consistent across the lifecycle.

## Preparing Data for Modeling
Tree-based models do not require feature scaling, but categorical variables need to be encoded.

In [ ]:
target = df["Churn"].map({"No": 0, "Yes": 1})

features = df.drop(columns=["customerID", "Churn"]).copy()
features = pd.get_dummies(features, drop_first=True)

X = features
y = target

print(X.shape, y.shape)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Modeling Utilities
To keep the evaluation consistent, we use a helper function that returns the main classification metrics for each model.

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds)
    }

    print(f"=== {model_name} ===")
    print(classification_report(y_test, preds))

    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"Confusion Matrix: {model_name}")
    plt.show()

    return results

# Model 1: Decision Tree
A single decision tree is highly interpretable and provides a useful baseline for understanding how tree-based logic behaves on this problem.

In [ ]:
dt_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_params,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)
print("Best Decision Tree params:", dt_grid.best_params_)

In [ ]:
dt_results = evaluate_model(
    dt_grid.best_estimator_,
    X_train, y_train, X_test, y_test,
    "Decision Tree"
)

In [ ]:
dt_importance = (
    pd.Series(dt_grid.best_estimator_.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=dt_importance.values, y=dt_importance.index)
plt.title("Top Decision Tree Feature Importances")
plt.show()

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    dt_grid.best_estimator_,
    feature_names=X.columns,
    class_names=["No Churn", "Churn"],
    filled=True,
    max_depth=2,
    fontsize=8
)
plt.title("Decision Tree (Top Levels)")
plt.show()

# Model 2: Random Forest
Random Forest reduces the instability of a single tree by averaging many trees trained on different bootstrap samples.

In [ ]:
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, None],
    "max_features": ["sqrt", "log2", None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
print("Best Random Forest params:", rf_grid.best_params_)

In [ ]:
rf_results = evaluate_model(
    rf_grid.best_estimator_,
    X_train, y_train, X_test, y_test,
    "Random Forest"
)

In [ ]:
rf_importance = (
    pd.Series(rf_grid.best_estimator_.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=rf_importance.values, y=rf_importance.index)
plt.title("Top Random Forest Feature Importances")
plt.show()

# Model 3: Gradient Boosting
Gradient Boosting sequentially improves weak learners and often performs well on structured tabular data.

In [ ]:
gb_params = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=gb_params,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)
print("Best Gradient Boosting params:", gb_grid.best_params_)

In [ ]:
gb_results = evaluate_model(
    gb_grid.best_estimator_,
    X_train, y_train, X_test, y_test,
    "Gradient Boosting"
)

In [ ]:
gb_importance = (
    pd.Series(gb_grid.best_estimator_.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=gb_importance.values, y=gb_importance.index)
plt.title("Top Gradient Boosting Feature Importances")
plt.show()

## Model Comparison
A compact comparison table makes it easier to decide which model is the strongest overall.

In [ ]:
results_df = pd.DataFrame([dt_results, rf_results, gb_results]).sort_values(
    by="f1", ascending=False
)
results_df

In [ ]:
results_df.set_index("model")[["accuracy", "precision", "recall", "f1"]].plot(
    kind="bar", figsize=(10, 5)
)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()

## Final Takeaways
Typical findings from this workflow usually show that:
- contract type is one of the strongest churn predictors
- early-tenure customers are often more likely to churn
- tree-based ensembles usually outperform a single decision tree
- feature importance can help connect model output back to business actions

## Portfolio Value
This project demonstrates:
- end-to-end supervised learning workflow
- structured EDA for a business problem
- feature engineering with interpretable cohorting
- model tuning with GridSearchCV
- classification metric comparison for churn prediction